# 06 · TS-target PCA — alpha and mean targets

**Purpose.** Find factors that explain a *time-series target* $G$ ($T\times J$).
With the **alpha target** the factors focus on what a reference model (e.g.
Fama-French 5) fails to price; with the **mean target** $G=1_T$ the estimator is
RP-PCA.

**Specification (Proposition 2).** Weights are the top-$K$ eigenvectors of
$\frac1{NT}R'(I_T+\gamma_{TS}P_G)R$, factors $F=R\hat\omega$, loadings by OLS
with intercept. Alpha target (eq. 10):
$G=(I_T-\tilde F(\tilde F'\tilde F)^{-1}F^{ref\prime})1_T$, so that $R'G/T$ are the
reference-model alphas.

| Parameter | Default | Options |
|---|---|---|
| `n_factors` | 1 | number of factors |
| `gamma_ts` | 15.0 | target weight $\ge-1$ (paper benchmark 15) |
| `ts_target` | `"alpha"` | `"alpha"`, `"mean"`, `"comove"` (extension), `"given"` |
| `publication_lag` | 0 | ignore the last periods of the reference factors |

### How to run this notebook

```bash
pip install -e ".[notebooks]"        # package + jupyter, matplotlib, pandas
jupyter lab examples/                 # or: jupyter notebook examples/
```

All data are **synthetic with known ground truth** (generated by
`lfm.simulate_factor_returns` / `lfm.simulate_ipca_panel`), so every cell runs
offline in seconds and estimates can be compared with the truth. To use your
own data, replace the simulation cell by a `pandas.DataFrame` of excess returns
(dates x assets); every estimator accepts DataFrames directly.

In [1]:
# Setup: imports, plotting defaults and reproducibility
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import latent_factor_models as lfm

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.figsize": (8, 3.6), "axes.grid": True, "grid.alpha": 0.3})
print("latent-factor-models", lfm.__version__)

latent-factor-models 0.1.0


In [2]:
sim = lfm.simulate_factor_returns(T=600, N=100, seed=0)
data = sim.to_data("reference")  # factors= reference model (a market proxy) for the alpha target

## 1. The alpha target reproduces reference-model alphas

In [3]:
from latent_factor_models.models.targets import alpha_target

G = alpha_target(sim.reference)
X = np.column_stack([np.ones(600), sim.reference])
alphas = np.linalg.lstsq(X, sim.returns, rcond=None)[0][0]
print("R'G/T equals the time-series alphas:", np.allclose(sim.returns.T @ G[:, 0] / 600, alphas))

R'G/T equals the time-series alphas: True


## 2. Effect of γ_TS — the TS-metric

TS-metric (eq. 16): $100\,\lVert P_GRP_{\hat\omega}\rVert_F/\lVert P_GR\rVert_F$, the share
of the target-projected returns (here: reference alphas) spanned by the factors.

In [4]:
rows = []
for g in [0, 1, 5, 15, 50]:
    m = lfm.TSTargetPCA(3, gamma_ts=g, ts_target="alpha").fit(data)
    rows.append({**m.evaluate(), "TS-metric": m.target_metrics()["ts_metric"]})
table = pd.DataFrame(rows, index=pd.Index([0, 1, 5, 15, 50], name="gamma_ts"))
table

,sharpe_ratio,rms_alpha,unexplained_variation,total_r2,TS-metric
gamma_ts,,,,,
0,0.3311,0.1288,0.3371,0.7060,93.9308
1,0.3394,0.1272,0.3371,0.7060,95.2224
5,0.3609,0.1237,0.3372,0.7059,97.8719
15,0.3814,0.1214,0.3374,0.7057,99.4177
50,0.3960,0.1202,0.3376,0.7055,99.9222


## 3. Target types side by side

In [5]:
specs = {
    "alpha target": lfm.TSTargetPCA(3, 15, "alpha"),
    "mean target (= RP-PCA)": lfm.TSTargetPCA(3, 15, "mean"),
    "co-movement (extension)": lfm.TSTargetPCA(3, 15, "comove"),
    "RP-PCA γ=15": lfm.RPPCA(3, gamma=15),
}
pd.DataFrame({k: m.fit(data).evaluate() for k, m in specs.items()}).T

,sharpe_ratio,rms_alpha,unexplained_variation,total_r2
alpha target,0.3814,0.1214,0.3374,0.7057
mean target (= RP-PCA),0.3814,0.1214,0.3374,0.7057
co-movement (extension),0.3311,0.1288,0.3371,0.7060
RP-PCA γ=15,0.3814,0.1214,0.3374,0.7057


## 4. Out of sample — no look-ahead

The alpha target depends on the reference factors' means; it is rebuilt from the
**training window only** in every rolling window and CV fold.
`publication_lag` additionally ignores the most recent reference observations.

In [6]:
engine = lfm.Engine(
    [
        lfm.PCA(3),
        lfm.TSTargetPCA(3, gamma_ts=15),
        lfm.TSTargetPCA(3, gamma_ts=15, publication_lag=3),
    ]
)
engine.out_of_sample(data, window=240, refit_every=6).to_frame().set_index("model")

,sample,sharpe_ratio,rms_alpha,unexplained_variation,total_r2,n_windows
model,,,,,,
PCA(n_factors=3),out-of-sample,0.2782,0.1749,0.3505,0.6935,60
TSTargetPCA(n_factors=3),out-of-sample,0.2944,0.1648,0.3512,0.6928,60
"TSTargetPCA(n_factors=3,publication_lag=3)",out-of-sample,0.2934,0.1651,0.3512,0.6928,60


In [7]:
# A user-defined target: pass ts_target=... and choose ts_target="given"
custom = np.sign(sim.reference - sim.reference.mean())  # e.g. up/down-market indicator
pd.Series(lfm.TSTargetPCA(3, 10, ts_target="given").fit(sim.returns, ts_target=custom).evaluate())

sharpe_ratio             0.3311
rms_alpha                0.1288
unexplained_variation    0.3371
total_r2                 0.7060
dtype: float64